<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/2026_09_25___Cleaning_Gauntlet___Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
raw = df.copy()  # snapshot of the generated data, used for the write-up revenue check

initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
duplicate_rows_dropped = initial_rows - df.shape[0]
log(1, 'Dropped exact duplicate rows', duplicate_rows_dropped)

[1] Dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
dollar_rows = df['price'].astype(str).str.contains('$', regex=False).sum()
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).astype(float)
assert df['price'].notna().all()  # conversion introduced no nulls
log(2, f"Stripped '$' from {dollar_rows} prices; cast all {df.shape[0]} to float", dollar_rows)

[2] Stripped '$' from 124 prices; cast all 300 to float (124 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
initial_qty_rows = df.shape[0]
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
n_missing = df['qty'].isna().sum()
n_negative = (df['qty'] < 0).sum()
df = df[df['qty'] >= 1].copy()  # NaN fails the comparison, so this drops both
qty_rows_dropped = initial_qty_rows - df.shape[0]
log(3, f'Dropped rows with missing qty ({n_missing}) or negative qty ({n_negative}); treated negatives as bad data, not returns', qty_rows_dropped)

[3] Dropped rows with missing qty (12) or negative qty (13); treated negatives as bad data, not returns (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

variants_before = df['item'].nunique()
items_changed = (~df['item'].isin(set(ITEM_MAP.values()))).sum()
df['item'] = df['item'].map(ITEM_MAP)
log(4, f"Canonicalized item names: {variants_before} spellings -> {df['item'].nunique()} products", items_changed)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4] Canonicalized item names: 6 spellings -> 3 products (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

# Decision: Apparel and Merch are one line of business here (non-food goods), so merge them.
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

spelling_fixes = df['category'].isin(['food', 'rain-gear']).sum()
apparel_rows = (df['category'] == 'Apparel').sum()
df['category'] = df['category'].map(CATEGORY_MAP)
log(5, f"Normalized category spellings ({spelling_fixes} rows); merged Apparel into Merch ({apparel_rows} rows)", spelling_fixes + apparel_rows)

# The source category column is not tied to the item: the same product appears under every category.
print(pd.crosstab(df['item'], df['category']))

# Decision: a product has one category, so derive category from item rather than trust the source column.
ITEM_CATEGORY = {'Cheeseburger': 'Food', 'Foam Finger': 'Merch', 'Rain Poncho': 'RainGear'}
source_category = df['category'].copy()  # kept for the write-up comparison
derived = df['item'].map(ITEM_CATEGORY)
category_overrides = (df['category'] != derived).sum()
df['category'] = derived
log(5, 'Replaced source category with item-derived category (source column contradicted item)', category_overrides)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[5] Normalized category spellings (89 rows); merged Apparel into Merch (43 rows) (132 row(s))
category      Food  Merch  RainGear
item                               
Cheeseburger    27     35        25
Foam Finger     39     30        28
Rain Poncho     29     29        33
[5] Replaced source category with item-derived category (source column contradicted item) (185 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}   # exactly three canonical products
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}                # no stray spellings, Apparel merged
assert df.groupby('item')['category'].nunique().max() == 1                          # each product sits in one category
assert df[['item', 'category', 'qty', 'price']].notna().all().all()                 # mapping left no NaNs
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print('Revenue by Category:')
print(revenue_by_category.round(2).map('${:,.2f}'.format).to_string())

total_revenue = df['revenue'].sum()
print(f'\nOverall Total Revenue: ${total_revenue:,.2f}')

log(7, 'Calculated revenue by category and total revenue', df.shape[0])

Revenue by Category:
category
RainGear    $1,729.50
Merch       $1,593.00
Food        $1,417.50

Overall Total Revenue: $4,740.00
[7] Calculated revenue by category and total revenue (275 row(s))


**What I would tell the vendor:** *Stock more rain ponchos: RainGear is the top category at \$1,729.50, about 36% of the \$4,740.00 total.*

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,Dropped exact duplicate rows,15
1,2,Stripped '$' from 124 prices; cast all 300 to ...,124
2,3,Dropped rows with missing qty (12) or negative...,25
3,4,Canonicalized item names: 6 spellings -> 3 pro...,126
4,5,Normalized category spellings (89 rows); merge...,132
5,5,Replaced source category with item-derived cat...,185
6,7,Calculated revenue by category and total revenue,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In [11]:
# Revenue at each stage, for write-up (a). Price is cleaned early on the raw frames so revenue is computable;
# .sum() skips the 12 NaN quantities. Price, item and category steps don't change the total.
def revenue(frame):
    price = frame['price'].astype(str).str.replace('$', '', regex=False).astype(float)
    return (pd.to_numeric(frame['qty'], errors='coerce') * price).sum()

deduped = raw.drop_duplicates()
stages = pd.Series({
    'raw (as generated)': revenue(raw),
    'after TODO 1 (dedup)': revenue(deduped),
    'after TODO 3 (qty filter)': revenue(deduped[deduped['qty'] >= 1]),
    'final (reported)': df['revenue'].sum(),
})
print(pd.DataFrame({'revenue': stages, 'change': stages.diff()}).round(2).to_string())

# For write-up (b): the same revenue grouped by the source category column instead
print('\nUsing source category column (Apparel merged into Merch):')
print(df.groupby(source_category)['revenue'].sum().sort_values(ascending=False).round(2).to_string())

                           revenue  change
raw (as generated)          4852.5     NaN
after TODO 1 (dedup)        4594.5  -258.0
after TODO 3 (qty filter)   4740.0   145.5
final (reported)            4740.0     0.0

Using source category column (Apparel merged into Merch):
category
Food        1656.0
Merch       1572.0
RainGear    1512.0


**a) Which cleaning step changed your revenue total the most?**

Dropping duplicates (TODO 1). Revenue went from 4,852.50 to 4,594.50, a change of −258.00. The qty filter (TODO 3) was second: 4,594.50 to 4,740.00, +145.50. It went up because the 13 negative-quantity rows were contributing negative revenue. The price, item, and category steps don't change the total.

**b) Pick one decision you made where a reasonable person could have chosen differently.**

I derived category from item (Cheeseburger → Food, Foam Finger → Merch, Rain Poncho → RainGear) instead of using the `category` column as given. The other choice is to trust the source column after fixing spellings and merging Apparel into Merch. That leaves the total unchanged at 4,740.00 but reverses the ranking: Food 1,656.00, Merch 1,572.00, RainGear 1,512.00, versus my RainGear 1,729.50, Merch 1,593.00, Food 1,417.50. The vendor advice would flip from "stock more ponchos" to "stock more food." I went with item-derived categories because the source column contradicts the item on most rows (only 27 of 87 cheeseburgers are tagged Food), so a category total built on it would not be a number I could defend.